In [ ]:
!pip install openai
!pip install tiktoken
!python -m pip install --upgrade pip
!pip install torch --upgrade --quiet
!pip install transformers --quiet
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install --upgrade transformers

In [ ]:
import os
from openai import AzureOpenAI

endpoint = <"Enter Endpoint here">
model_name = "gpt-4.1"
deployment = "gpt-4.1"

subscription_key = <"ENter subscription key here">
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "I am going to Paris, what should I see?",
        }
    ],
    max_completion_tokens=800,
    temperature=1.0,
    top_p=1.0,
    frequency_penalty=0.0,
    presence_penalty=0.0,
    model=deployment
)

print(response.choices[0].message.content)

# Practicals: Hands-on with LLMs
In this section, you will find practical exercises to try out key LLM concepts. Each exercise includes a theory cell and a code cell. Run the code cells and observe the results. Modify the inputs and parameters to deepen your understanding.

## 1. Tokenization Exercise
Tokenize different sentences and compare their token counts. Try your own sentences! Tokenization is how LLMs break text into pieces (tokens) for processing. Understanding token counts is important for prompt design and cost estimation.

In [ ]:
# Tokenization code: Run this cell to see how sentences are tokenized
import tiktoken
encoding = tiktoken.get_encoding('cl100k_base')  # Use the base encoding for GPT-4 models
sentences = [
    "The quick brown fox.",
    "Large Language Models are powerful.",
    "Try your own sentence here!"
 ]
for s in sentences:
    tokens = encoding.encode(s)
    print(f"Text: {s}")
    print(f"Tokens: {tokens}")
    print(f"Token count: {len(tokens)}\n")

In [ ]:
# Detailed token breakdown for Azure OpenAI chat completion request



system_prompt = "You are a helpful assistant."
user_message = "I am going to Paris, what should I see?"
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_message}
 ]

encoding = tiktoken.get_encoding('cl100k_base')

system_tokens = len(encoding.encode(system_prompt))
user_tokens = len(encoding.encode(user_message))

print(f"System prompt: {system_prompt}")
print(f"System prompt tokens: {system_tokens}\n")
print(f"User message: {user_message}")
print(f"User message tokens: {user_tokens}\n")

response = client.chat.completions.create(
    messages=messages,
    max_completion_tokens=800,
    temperature=1.0,
    top_p=1.0,
    frequency_penalty=0.0,
    presence_penalty=0.0,
    model=deployment
)

response_text = response.choices[0].message.content
response_tokens = len(encoding.encode(response_text))

# print(f"Response: {response_text}")
print(f"Response tokens: {response_tokens}")

## 2. Embedding Similarity
Embeddings are vector representations of text that capture semantic meaning. You can compare the similarity of two texts by calculating the cosine similarity between their embeddings. Try with similar and different sentences to see how the similarity score changes.

In [ ]:
# Set up Azure OpenAI embedding client for text-embedding-3-large model
from openai import AzureOpenAI

embedding_model_name = "text-embedding-3-large"  # Model name for embeddings
embedding_deployment = "text-embedding-3-large"  # Deployment name for embeddings
embedding_api_version = "2024-12-01-preview"
embedding_client = AzureOpenAI(
    api_version=embedding_api_version,
    azure_endpoint="<Enter Endpoint here>",
    api_key=<"Enter API Key Here">
)

In [ ]:
# This cell demonstrates how to generate embeddings for two sentences using Azure OpenAI's embedding_client, and then compute their semantic similarity using cosine similarity.
# You can modify the 'texts' list to compare other sentences. Higher cosine similarity means the sentences are more semantically similar.
# Embedding Similarity code: Run this cell to compare sentence embeddings using the embedding_client
import numpy as np
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

texts = [
    "Artificial intelligence is fascinating.",
    "AI is very interesting.",
 ]
# Use the embedding_client and embedding_deployment for embeddings
embeddings = embedding_client.embeddings.create(input=texts, model=embedding_deployment)
vec1 = embeddings.data[0].embedding
vec2 = embeddings.data[1].embedding

# Show the embedding vectors for each sentence
print("Embedding for first sentence:")
print(vec1)
print(f"Length of embedding vector: {len(vec1)}\n")
print("Embedding for second sentence:")
print(vec2)
print(f"Length of embedding vector: {len(vec2)}\n")



In [ ]:
# These embeddings are high-dimensional vectors (typically 1536 floats for OpenAI models) representing the semantic meaning of the sentences.
# Now calculate and print the cosine similarity
sim = cosine_similarity(vec1, vec2)
print(f"Cosine similarity: {sim}")

## 3. Prompt Engineering Practice
Prompt engineering is about crafting the right instructions and examples to guide the LLM. Zero-shot prompts give only the instruction, while few-shot prompts include examples. Try changing the instructions and examples to see how the model's response changes.

## Zero-shot vs Few-shot Prompting

**Zero-shot prompting** means you only give the model an instruction, with no examples. The model must figure out what to do from the instruction alone.

**Few-shot prompting** means you provide the model with a few examples of the task, so it can learn the pattern and respond accordingly. This often improves accuracy for more complex tasks.

Below, you can try both approaches and compare the results.

In [ ]:
# Zero-shot Prompt Example using AzureOpenAI client

messages = [
    {"role": "user", "content": "Translate 'Good morning' to French."}
 ]

response = client.chat.completions.create(
    messages=messages,
    max_completion_tokens=10,
    model=deployment
)
print("Zero-shot response:", response.choices[0].message.content)

In [ ]:
# Few-shot Prompt Example using AzureOpenAI client


messages = [
    {"role": "system", "content": "You are a helpful translator."},
    {"role": "user", "content": "Translate 'Hello' to French."},
    {"role": "assistant", "content": "Bonjour"},
    {"role": "user", "content": "Translate 'Goodbye' to French."},
    {"role": "user", "content": "Translate 'Good morning' to French."}
 ]

response = client.chat.completions.create(
    messages=messages,
    max_completion_tokens=10,
    model=deployment
)
print("Few-shot response:", response.choices[0].message.content)

## 4. Generation Parameters Experiment
LLMs have parameters like temperature and max tokens that affect their output. Temperature controls randomness (higher = more creative), and max tokens limits output length. Experiment with these to see how the model's responses change.

## Understanding Temperature and Max Tokens

- **Temperature** controls the randomness of the model's output. Lower values (e.g., 0.2) make the output more focused and deterministic, while higher values (e.g., 1.2) make it more creative and random.
- **Max tokens** sets the maximum length of the model's response. Increasing this allows for longer answers, while decreasing it limits the response length.

Below are examples showing how changing these parameters affects the model's output.

In [ ]:
# Experimenting with Temperature and Max Tokens using AzureOpenAI client
prompt = "Describe the future of AI in one sentence."
for temp in [0.2, 0.7, 1.2]:
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=30,
        temperature=temp,
        model=deployment,
    )
    print(f"Temperature: {temp}")
    print(response.choices[0].message.content)
    print("-"*30)


In [ ]:

print("\n--- Varying Max Tokens (temperature = 0.7) ---")
for max_tokens in [10, 30, 60]:
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=max_tokens,
        temperature=0.7,
        model=deployment,
    )
    print(f"Max tokens: {max_tokens}")
    print(response.choices[0].message.content)
    print("-"*30)

# Hugging Face Transformers: Core Tasks

This section demonstrates five core NLP tasks using Hugging Face's `transformers` library. Each task includes a brief explanation and a runnable code cell.

---

## 1. Text Generation
Use a pre-trained model (like GPT-2 or DistilGPT-2) to generate text from a prompt.

In [ ]:
# Text Generation with GPT-2

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = 'gpt2'  # You can also try 'distilgpt2' for a smaller model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

prompt = 'Once upon a time, in a land far away,'
input_ids = tokenizer.encode(prompt, return_tensors='pt')
output = model.generate(input_ids, max_length=50, num_return_sequences=1, do_sample=True, temperature=0.8)
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

## 2. Text Classification (Sentiment Analysis)
Use a model (like DistilBERT) to classify the sentiment of a sentence.

In [ ]:
from transformers import pipeline
classifier = pipeline('sentiment-analysis')
sentence = 'I love using Hugging Face Transformers!'
result = classifier(sentence)
print(f'Sentence: {sentence}')
print(f'Sentiment: {result[0]["label"]}, Score: {result[0]["score"]:.2f}')

## 3. Named Entity Recognition (NER)
Use a model to extract entities (names, places, etc.) from a sentence.

In [ ]:
from transformers import pipeline
ner = pipeline('ner', grouped_entities=True)
sentence = 'Barack Obama was born in Hawaii and was the 44th President of the United States.'
entities = ner(sentence)
print(f'Sentence: {sentence}')
print('Entities:')
for entity in entities:
    print(f' - {entity["word"]}: {entity["entity_group"]} (score: {entity["score"]:.2f})')

## 4. Summarization
Use a model (like T5 or BART) to summarize a long piece of text.

In [ ]:
from transformers import pipeline
summarizer = pipeline('summarization')
text = ("Machine learning is a field of artificial intelligence that uses statistical techniques "
        'to give computer systems the ability to learn from data, without being explicitly programmed. '
        'It is seen as a part of artificial intelligence. Machine learning algorithms build a model '
        'based on sample data, known as training data, in order to make predictions or decisions '
        'without being explicitly programmed to do so.' )
summary = summarizer(text, max_length=40, min_length=10, do_sample=False)
print('Original text:', text)
print('Summary:', summary[0]['summary_text'])

## 5. Translation
Use a model (like Helsinki-NLP/opus-mt) to translate text from one language to another.

In [ ]:
from transformers import pipeline
translator = pipeline('translation_en_to_fr', model='Helsinki-NLP/opus-mt-en-fr')
text = 'Artificial intelligence is transforming the world.'
translation = translator(text, max_length=40)
print(f'Original text: {text}')
print(f'French translation: {translation[0]["translation_text"]}')

 # Demo: Medical Chatbot with Azure OpenAI

This example demonstrates a simple medical chatbot using Azure OpenAI. The chatbot will follow a specific set of instructions to answer basic medical questions, but will not provide diagnoses or replace professional advice.

In [ ]:


system_prompt = (
    "You are MedBot, a helpful and knowledgeable medical assistant. "
    "You can answer general health questions, explain symptoms, and provide information about medications, "
    "but you do not diagnose conditions or give medical advice. Always recommend consulting a healthcare professional for serious concerns."
)

print("Type 'quit' to end the chat.\n")

while True:
    user_input = input("User: ")
    if user_input.lower() in ["quit", "exit"]:
        print("MedBot: Take care! If you have a medical emergency, contact a professional.")
        break
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]
    response = client.chat.completions.create(
        messages=messages,
        max_completion_tokens=300,
        temperature=0.5,
        model=deployment
    )
    print("MedBot:", response.choices[0].message.content.strip())